# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets, fields, and columns are referenced by their `@id` fields. We enumerate the available record sets and their fields below.

In [ ]:
# List all record sets and their fields by @id
print("Available RecordSets in the dataset:")
record_sets = []
for rs in metadata.record_sets:
    print(f"  RecordSet name: {rs.name}, @id: {rs.id}")
    record_sets.append(rs.id)
    if hasattr(rs, 'fields'):
        print("    Fields (@id):")
        for field in rs.fields:
            print(f"      - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    else:
        print("    No fields defined.")

## 2.1 Example Record Preview
Let's print a few records from each record set by referencing them by their `@id`.

In [ ]:
for rs_id in record_sets:
    print(f"\nRecords from RecordSet @id: {rs_id}")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 1:
            print("...\n")
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into dataframes
dataframes = {}
for rs_id in record_sets:
    print(f"Loading data from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns in dataframe: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())
    else:
        print("No records loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will include filtering outliers, transforming numeric fields, and grouping.

In [ ]:
# For demonstration, select the first record set with data for EDA
selected_rs_id = None
for rs_id in dataframes:
    if not dataframes[rs_id].empty:
        selected_rs_id = rs_id
        break
if selected_rs_id is None:
    raise ValueError('No record set with records found for EDA!')

df = dataframes[selected_rs_id].copy()

# Preview columns and try to find a numeric field by @id for demonstration
print("Columns in selected record set:")
print(df.columns.tolist())

# Try to guess a numeric field by checking dtypes or common field names
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if not numeric_candidates:
    # Try casting all columns to numeric (ignore errors)
    for col in df.columns:
        df[col+'_tmp'] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = [col for col in df.columns if df[col].dtype in [float, int] and df[col].notnull().any()]

if not numeric_candidates:
    print('No numeric columns found for analysis.')
else:
    # Use the first candidate
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")

    # Set a threshold for filtering
    threshold = df[numeric_field].mean() if df[numeric_field].dtype in [float, int] else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find a group_field (categorical) for grouping
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object and df[col].nunique() <= 10:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print('No suitable group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_candidates:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {selected_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot grouped by group_field if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* We loaded the dataset using `mlcroissant` and listed available RecordSets, all referenced via their `@id` for reproducibility.
* We selected a record set and identified numeric and potential group fields for analysis.
* Common exploratory data analysis operations were performed: filtering, normalization, grouping, and basic plotting.

To further analyze this dataset, you may explore relationships between protein attributes, modifications, and sample conditions, referencing each field and record set always by their `@id`.

> For more details on working with Croissant datasets and their field organization, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).